# Argon Gas/Liquid Simulation: Part 2
> NAME: Harrison B. Prosper<br>
> DATE: March 2026 

## Learning objective

To learn how to simulate a cloud of interacting neutral argon atoms confined to  a **spherical container**. 

## Tips

  * Use __esc r__ to disable a cell
  * Use __esc y__ to reactivate it
  * Use __esc m__ to go to markdown mode. **Markdown** is the typesetting language used in jupyter notebooks.
  * In a markdown cell, double tap the mouse or glide pad (on your laptop) to go to edit mode. 
  * Shift + return to execute a cell (including markdown cells).
  * If the equations don't typeset, try double tapping the cell again, and re-execute it.

In [1]:
import os, sys

# Array manipulation and linear algebra
import numpy as np

# 3D animation system
import vpython as vp

from comphyslab.graphics import Sim, Controls, \
Scene, Zoom, CoordinateSystem, I, J, K

from comphyslab.vectors import unit, magnitude
from comphyslab.newton import argon_initial_state, propagate_order3
from comphyslab.utils import Bag, SimLogger

/Users/harry/miniconda3/envs/comphys/lib/python3.13/site-packages/vpython/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

## Intersection of a line with a sphere

The equation of a straight line in 3D space is given by
\begin{align}
    \vec{r} & = \vec{r}_0 + t \hat{u}.\tag{1}
\end{align}

The equation of a sphere of radius $d$, centered at the origin, is simply
\begin{align}
    r = d, \tag{2}
\end{align}
where $r = ||\vec{r}||$, that is, the magnitude of $\vec{r}$.

![sphere/line intersection](sphere_line.png)

**Not graded**

Derive an expression for the two intersection points of a line with a sphere by solving Eqs.(1) and (2) *simultaneously*.

*Hint*: Take the dot product of Eq.(1) with itself, then use Eq.(2).

In [2]:
def reflect(v, n):
    '''
    v:    Velocity of particles, shape (N,3) with N >= 1
    n:    Unit vector defining orientation of plane, shape (3,)
    '''
    # vdotn must be of the same dimensions as v and n, that is, a 2D array.
    # We use broadcasting to achieve this.
    
    many  = len(v) > 1
    vdotn = (v*n).sum(axis=-1)[:, None] if many else  (v*n).sum(axis=-1) 

    u = v - 2 * vdotn * n  # (N,3)
    
    return u

In [3]:
def line_sphere_point(c, u, d):
    '''
    c:    Fixed point on line(s), shape (N,3)
    u:    Unit vector defining the direction of line(s), shape (N,3)
    d:    Radius of sphere centered at origin
    '''
    cc = (c*c).sum(axis=-1) # c dot c
    uc = (u*c).sum(axis=-1) # u dot c
    
    s  = uc**2 - cc + d**2
    
    # s >= 0 if the line crosses the sphere
    crosses = s >= 0
 
    # Two possible intersection points defined by the two
    # scalars t1 and t2. Protect against possible tiny negative 
    # values due to floating point rounding errors.
    q = np.sqrt(np.clip(s, 0, None))
    t1 =-uc + q
    t2 =-uc - q

    # Since we are within the sphere, we choose the greater of t1 and t2    
    t = np.where(t1 > t2, t1, t2)

    # vdotn must be of the same dimensions as v and n, that is, a 2D array.
    # We use broadcasting to achieve this.
    is2D = len(c) > 1
    b = c + t[:, None] * u if is2D else c + t * u
    return b

In [4]:
def build_scene(bg):
    # --------------------------------------------------------
    # Required parameters
    # --------------------------------------------------------
    bg.rate   = 30         # Maximum frames/second
    bg.frame  = 0          # Frame counter
    bg.active = True       # Controls stopping of event loop
    bg.update = False      # Controls start/pause of update
    # --------------------------------------------------------
    bg.size = 1.05 * bg.R  # Characteristic size of scene (units of sigma) 
    bg.atom_radius = 0.1*bg.R

    # IMPORTANT: All scene widgets must go into bag.gfx
    gfx = bg.gfx
    
    # Create an empty scene
    gfx.scene = Scene('Argon Atoms\n', bg.size, height=200)

    # Create a Cartesian coordinate system with z-axis upwards
    gfx.xyz = CoordinateSystem(bg.size, up=K, draw_plane=False)

    # --------------------------------------------------------
    # Create a translucent sphere of radius bg.R,
    # centered at the origin. 
    #
    # Call the container 
    #
    #   gfx.container = ...
    # --------------------------------------------------------
    # CODE GOES HERE
    

    # --------------------------------------------------------
    # Create one or more atoms, modeled as small spheres.
    # Add each sphere to list gfx.atoms using 
    #
    #   gfx.atoms.append( ... )
    # --------------------------------------------------------
    gfx.atoms = [] # empty list
    
    #for i in range(bg.N):
        # CODE GOES HERE
        
    # --------------------------------------------------------
    # Create control buttons Stop, Start/Pause and a Zoom
    # slider.
    # --------------------------------------------------------
    controls = Controls(bg)

    gfx.b_stop  = vp.button(
        text="Stop",
        background=vp.color.red,
        pos=gfx.scene.title_anchor,
        bind=controls.stop)

    gfx.b_start_pause = vp.button(
        text="Start",
        background=vp.color.green,
        pos=gfx.scene.title_anchor,
        bind=controls.start_pause)

    gfx.zoom_slider = Zoom(gfx.scene)

# Exercise 6 [10 pt]

Complete the function `bounce_off_wall`. 
 
But first try to understand the code snippets in the cell below. 

### README! You may find the following tips useful
  1. Distinguish between the external name of the `Bag` object, which we have called `bag`, and the name used internally, `bg`, by `build_scene`.
  2. From now on, we shall be using `numpy` for most vector operations. But `vpython` and `numpy` represent vectors differently, so be aware of the differing syntax. 
  1. `bg.r` is a numpy array of shape (N,3). It represents N position vectors, while `bg.v` represents N velocities.
  2. `bg.r[i]` accesses row $i$ of `bg.r`, that is, the position of atom $i$.
  3. The `magnitude` function computes the magnitude of one or more vectors.
  4. The `unit` function computes unit vectors from one of more vectors.

In [5]:
# 1.
a = np.array([3.0,-3.0,5.0,-2.0,7.0])
b = np.zeros_like(a) # Make an array b like a but filled with zeros

b[:] = a    # What's the difference between the operation b[:] = a and c = a?
c = a

c[3] = -42
b[3] = -99

print('1')
print('a', a)
print('b', b)
print('c', c)
print()

# 2.
print('2')
mask = a > 0
print('mask', mask)
print()

# 3.
print('3')
c = a[mask]
print('c', c)
print()

# 4.
print('4')
b[mask] = np.sqrt(c)
print('b', b)

1
a [  3.  -3.   5. -42.   7.]
b [  3.  -3.   5. -99.   7.]
c [  3.  -3.   5. -42.   7.]

2
mask [ True False  True False  True]

3
c [3. 5. 7.]

4
b [  1.73205081  -3.           2.23606798 -99.           2.64575131]


In [6]:
def bounce_off_wall(bg, eps=1.e-8):
    # ----------------------------------------------
    # Determine which atoms must bounce.
    #    Hint: create an array of booleans, which 
    #    should be called "bounce":
    #
    #    bounce = <some expression>
    # ----------------------------------------------
    # Compute distances from the center of the
    # container.
    rmag  = magnitude(bg.r)

    # Compute boolean array "bounce"
    # 1. [1pt] CODE HERE

    # ----------------------------------------------
    # Determine bounce points and implement bounce.
    #    The boolean array "bounce" makes it possible
    #    to access only those elements of the arrays
    #    bg.r and bg.v that correspond to the atoms
    #    that need to bounce.
    #
    # Syntax:
    #    bg.v[bounce] = ...
    #
    # Hint: See code snippets in the cell above to
    #    understand exactly what the above does.
    # ----------------------------------------------
    if np.any(bounce): # <= What do you think this does?

        # Get current positions of atoms to bounce
        r0= bg.r0[bounce]                    # (k,3), k <= N

        # Get predicted positions and velocities of atoms to bounce
        r = bg.r[bounce]                     # (k,3)
        v = bg.v[bounce]                     # (k,3)

        # Compute unit vectors from r0 to r
        # 2. [1pt] CODE HERE

        # Compute bounce points b
        # 3. [1pt] CODE HERE

        # Compute unit vectors of tangent planes at bounce points.
        # 4. [1pt] CODE HERE

        # Relect velocity vector
        # 5. [1pt] CODE HERE

        # Compute vector from bounce points b to predicted points r
        # 6. [1pt] CODE HERE

        # Compute distance from bounce points b to predicted points r
        # 7. [1pt] CODE HERE

        # Update bg.r and bg.v, but only the atoms that have bounced!
        # Hint:
        #   bg.r[bounce] = <some expression>
        #   bg.v[bounce] = <some expression>
        # 8. [3pt] CODE HERE

## Predict the positions and velocities of all argon atoms

In [7]:
def propagate(bg):
    # ----------------------------------------------
    # Update state of every particle.
    # ----------------------------------------------
    bg.r0[:] = bg.r # copy r into r0 before update
    bg.v0[:] = bg.v # copy v into v0 before update

    # Solve Newton's 2nd law for every atom
    bg.r[:], bg.v[:], bg.U = propagate_order3(
        bg.k, bg.q, bg.m, bg.r0, bg.v0, bg.law, bg.dt)

## Update positions of all argon atoms

In [8]:
def update(bg):
    # ----------------------------------------------
    # 1. Compute next positions
    # ----------------------------------------------
    propagate(bg)

    # ----------------------------------------------
    # 2. Bounce atoms off wall as needed. 
    # ----------------------------------------------
    #bounce_off_wall(bg)
    
    # ----------------------------------------------
    # 3. Loop over graphical objects and update
    #    their positions. 
    # ----------------------------------------------
    for i, atom in enumerate(bg.gfx.atoms):
        x, y, z = bg.r[i]
        atom.pos.x = x
        atom.pos.y = y
        atom.pos.z = z

    if bag.save:
        bag.logger.write(bg)

## Define initial state of system

### Argon atom properties

| **Quantity** | **Symbol** | **Value** |
|----------|----------|----------|
| Mass     | $m$       | $6.69\times 10^{-26}\,\text{kg}$ |
| Distance scale | $\sigma$    | $3.40\times 10^{-10}\,\text{m}$  |
| Characteristic time scale     | $t_c$       | $2.16\times 10^{-12}\,\text{s}$  |
| Characteristic speed    | $v_c$       | $157.38\, \text{m/s}$   |

# Extra Credit [2 pt]

$N = 297$ argon atoms are placed in a spherical container of radius $R$ so that the mass density of atoms is $\rho = 1600 \text{ kg}/\text{m}^3$. Compute the radius, $R$, in meters and in units of $\sigma$.

### Conditions

  * Temperature: $T = 273\,\text{K}$ (0 Celcius).
  * density: $\rho = 50.0\text{ kg}/\text{m}^3$
  * Average speed of atoms: $2.6$ in units of $v_c$.
  * Container radius: $R = 13.414$ in units of $\sigma$.

In [9]:
rho = 50.0 # kg/m^3
T   = 273  # K (kelvin)
save_every = 30

bag = argon_initial_state(rho, T)

# Create a logger object to log simulation data to file

# These quantities are written out once
attributes = [
    'mass',      # Mass of argon atom (kg)
    'epsilon',   # Characteristic energy (J)
    'tc',        # Characteristic time scale (s)
    'vc',        # Characteristic speed (m/s)
    'sigma',     # Characteristic distance scale (m)
    'equi_sep',  # Equilibrium separation (sigma)

    'rho',       # Number density (number / sigma^3)
    'T',         # Temperature (K)

    'N',         # Number of atoms
    'R',         # Radius of container
    'rmin_sep',  # Minimum separation of atoms
    
    'dt',        # Simulation time step (tc)
    
    'T2K', 'vrms'
]

# These quantities are written out periodically
datasets = [
    'r',         # Positions of atoms   (sigma)
    'v',         # Velocities of atoms  (vc)
    'impulse',   # Momentum change imparted to container wall (mass*vc)
    'U'          # Potential energy of atoms (epsilon)
]


--------------------------------------------------------------------------
Attributes (saved once)
--------------------------------------------------------------------------
  mass:      6.690e-26 	Mass of argon atom (kg)
  epsilon:   1.657e-21 	Characteristic energy (J)
  tc:        2.160e-12 	Characteristic time scale (s)
  vc:        1.574e+02 	Characteristic speed (m/s)
  sigma:     3.400e-10 	Characteristic distance scale (m)
  equi_sep:  1.123e+00 	Equilibrium separation (sigma)
  
  rho:       2.938e-02 	Number of atoms/sigma^3
  T:         2.730e+02 	Absolute temperature (K)

  N:               297 	Number of atoms in container
  R:         1.341e+01 	Radius of container (sigma)
  rmin_sep:  3.250e+00 	Minimum atomic separation (sigma)
  
  dt:        1.000e-03 	Simulation time step (tc)
--------------------------------------------------------------------------
Datasets (saved periodically)
--------------------------------------------------------------------------
  r: 			Init

In [10]:
filename = 'argon_sim.h5'

bag.save = False

if bag.save:
    bag.logger = SimLogger(
        filename, 
        bag, 
        attributes, datasets, 
        save_every)

build_scene(bag)

sim = Sim(bag, update)

sim.run()

if bag.save:
    bag.logger.close()

<IPython.core.display.Javascript object>

Animation ended!
